## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(viridis)
library(ggrepel)

# Read in Data

In [4]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Panther_2016", "Reactome_2022", "RNAseq_DiseaseGene_DrugSigs_GEO")

In [5]:
# Initialize an empty dataframe
all_data <- data.frame()

# Specify the directory containing the files
directory <- "06_DEG_GO"

# List all files in the directory that match the pattern
file_list <- list.files(path = directory, pattern = "results_enrichr_results\\.xlsx$", full.names = TRUE)

# Iterate over each file and read in the corresponding Excel sheets
for (file_path in file_list) {
  # Extract the comparison name from the file name
  comparison <- tools::file_path_sans_ext(basename(file_path))
      
  # Get the names of the sheets in the Excel file
  databases <- excel_sheets(file_path)
  
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file_path, sheet = database)
      
      if (nrow(df) == 0) {
        next
      }
      
      # Select the required columns and add module (comparison) and database information
      df <- df %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = comparison, Database = database)

      # Filter out rows where the Adjusted.P.value is less than or equal to 0.05
      df <- df %>%
      filter(Adjusted.P.value <= 0.01)
        
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Rename
all_data$Module <- gsub("_results_enrichr_results$", "", all_data$Module)

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Chemical Synaptic Transmission (GO:0007268),9.217457e-05,4.115653,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025
2,Neuron Projection Guidance (GO:0097485),3.988211e-03,4.731604,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025
3,Axon Guidance (GO:0007411),3.988211e-03,4.371209,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025
4,Modulation of Chemical Synaptic Transmission (GO:0050804),3.988211e-03,4.978439,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025
5,Neuron Development (GO:0048666),4.887971e-03,4.157216,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025
6,Synaptic Vesicle Cycle (GO:0099504),4.953296e-03,8.553285,MUT_Guide6_vs_MUT_NonEffector,GO_Biological_Process_2025


In [6]:
# Subset for human 
all_human_data <- all_data

In [7]:
# Count terms for human data
human_term_counts <- all_human_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term),
    .groups = "drop"
  )

# Print results
cat("=== Human Data Term Counts ===\n")
print(human_term_counts)

=== Human Data Term Counts ===
# A tibble: 7 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 GO_Biological_Process_2025              201          137
2 GO_Cellular_Component_2025               34           20
3 GO_Molecular_Function_2025               53           38
4 KEGG_2021_Human                          25           19
5 Panther_2016                              6            6
6 RNAseq_DiseaseGene_DrugSigs_GEO         117           85
7 Reactome_2022                            96           66


# Top 50 Dot Plots

In [8]:
# === HUMAN DATA ===
human_term_module_counts <- all_human_data %>%
  group_by(Database, Term) %>%
  summarise(ModuleCount = n_distinct(Module), .groups = 'drop')

human_ranked_terms <- human_term_module_counts %>%
  arrange(Database, desc(ModuleCount)) %>%
  group_by(Database) %>%
  slice_head(n = 50) %>%
  ungroup()

human_filtered_top_50 <- all_human_data %>%
  semi_join(human_ranked_terms, by = c("Database", "Term"))

# Print results
cat("=== Human Top 50 Filtered Data ===\n")
print(head(human_filtered_top_50))

=== Human Top 50 Filtered Data ===
                                                       Term Adjusted.P.value
1               Chemical Synaptic Transmission (GO:0007268)     9.217457e-05
2                   Neuron Projection Guidance (GO:0097485)     3.988211e-03
3                                Axon Guidance (GO:0007411)     3.988211e-03
4 Modulation of Chemical Synaptic Transmission (GO:0050804)     3.988211e-03
5                           Neuron Development (GO:0048666)     4.887971e-03
6                      Potassium Ion Transport (GO:0006813)     5.583309e-03
  Odds.Ratio                        Module                   Database
1   4.115653 MUT_Guide6_vs_MUT_NonEffector GO_Biological_Process_2025
2   4.731604 MUT_Guide6_vs_MUT_NonEffector GO_Biological_Process_2025
3   4.371209 MUT_Guide6_vs_MUT_NonEffector GO_Biological_Process_2025
4   4.978439 MUT_Guide6_vs_MUT_NonEffector GO_Biological_Process_2025
5   4.157216 MUT_Guide6_vs_MUT_NonEffector GO_Biological_Process_2025
6   4.

In [9]:
# Clean Module names in both datasets
human_filtered_top_50 <- human_filtered_top_50 %>%
  mutate(Module = gsub("(Human|Mouse)", "", Module),
         Module = gsub("_", " ", Module))

# === HUMAN FILTERED DATA ===
human_filtered_term_counts <- human_filtered_top_50 %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term),
    .groups = "drop"
  )

# Print results
cat("=== Human Filtered Top-50 Term Counts ===\n")
print(human_filtered_term_counts)

=== Human Filtered Top-50 Term Counts ===
# A tibble: 7 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 GO_Biological_Process_2025              107           50
2 GO_Cellular_Component_2025               34           20
3 GO_Molecular_Function_2025               53           38
4 KEGG_2021_Human                          25           19
5 Panther_2016                              6            6
6 RNAseq_DiseaseGene_DrugSigs_GEO          82           50
7 Reactome_2022                            80           50


In [12]:
create_filtered_dot_plots <- function(data, species, output_dir = "00_FIGURES/Dot_Plots") {
  # Create output directory if it doesn't exist
  dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
  
  for (database in unique(data$Database)) {
    database_filtered_data <- data %>%
      filter(Database == database)
    
    if (nrow(database_filtered_data) > 0) {
      dot_plot <- ggplot(database_filtered_data, aes(x = Module, y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
        geom_point(shape = 21) +
        scale_fill_viridis() +
        xlab('') + ylab('') +
        labs(
          title = 'Top GO Terms Associated with Significant DEGs',
          subtitle = glue('{species} - {database}')
        ) +
        theme(
          panel.background = element_rect(fill = "white", color = NA),
          panel.grid.major = element_blank(),
          panel.grid.minor = element_blank(),
          panel.border = element_blank(),
          axis.line = element_line(color = "black"),
          plot.background = element_rect(fill = "white", color = NA),
          axis.text.x = element_text(angle = 90, hjust = 1),
          axis.text.y = element_text(size = 8)
        )
      
      # Save with species and database in filename
      filename <- glue("{output_dir}/top_50_dot_plot_{species}_{database}.pdf")
      ggsave(filename = filename, plot = dot_plot, height = 11, width = 10, limitsize = FALSE)
      cat(glue("Saved: {filename}\n"))
      
    } else {
      cat(glue("No data available for {species} - {database}. Skipping...\n"))
    }
  }
}

In [13]:
# Generate dot plots
create_filtered_dot_plots(human_filtered_top_50, species = "Human")

Saved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_GO_Biological_Process_2025.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_GO_Cellular_Component_2025.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_GO_Molecular_Function_2025.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_KEGG_2021_Human.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_Reactome_2022.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_RNAseq_DiseaseGene_DrugSigs_GEO.pdfSaved: 00_FIGURES/Dot_Plots/top_50_dot_plot_Human_Panther_2016.pdf